In [ ]:
#Now, we'll cover how to actually deploy our agent locally to Studio and to LangGraph Cloud.


In [1]:
#There are a few central concepts to understand -

#LangGraph —

#Python and JavaScript library
#Allows creation of agent workflows
#LangGraph API —

#Bundles the graph code
#Provides a task queue for managing asynchronous operations
#Offers persistence for maintaining state across interactions
#LangSmith Deployment (formerly LangGraph Cloud) --

#Hosted service for the LangGraph API
#Allows deployment of graphs from GitHub repositories
#Also provides monitoring and tracing for deployed graphs
#Accessible via a unique URL for each deployment
#LangSmith Studio (formerly LangGraph Studio) --

#Integrated Development Environment (IDE) for LangGraph applications
#Uses the API as its back-end, allowing real-time testing and exploration of graphs
#Can be run locally or with cloud-deployment. See below.
#LangGraph SDK --

#Python library for programmatically interacting with LangGraph graphs
#Provides a consistent interface for working with graphs, whether served locally or in the cloud
#Allows creation of clients, access to assistants, thread management, and execution of runs

from langgraph_sdk import get_client

In [3]:
# This is the URL of the local development server
URL = "http://127.0.0.1:2024"
client = get_client(url=URL)

# Search all hosted graphs
assistants = await client.assistants.search()

In [4]:
assistants[-3]

{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
 'graph_id': 'agent',
 'config': {},
 'context': {},
 'metadata': {'created_by': 'system'},
 'name': 'agent',
 'created_at': '2026-04-03T06:55:39.434298+00:00',
 'updated_at': '2026-04-03T06:55:39.434298+00:00',
 'version': 1,
 'description': None}

In [5]:
# We create a thread for tracking the state of our run
thread = await client.threads.create()

In [6]:
#Now, we can run our agent with client.runs.stream with:

#The thread_id
#The graph_id
#The input
#The stream_mode
#We'll discuss streaming in depth in a future module.

#For now, just recognize that we are streaming the full value of the state after each step of the graph with stream_mode="values".

#The state is captured in the chunk.data.

In [7]:
from langchain_core.messages import HumanMessage

# Input
input = {"messages": [HumanMessage(content="Multiply 3 by 2.")]}

# Stream
async for chunk in client.runs.stream(
        thread['thread_id'],
        "agent",
        input=input,
        stream_mode="values",
    ):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])

{'content': 'Multiply 3 by 2.', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': 'c0b9b807-1f13-404e-a995-58abb0264aed'}
{'content': '', 'additional_kwargs': {'refusal': None}, 'response_metadata': {'token_usage': {'completion_tokens': 17, 'prompt_tokens': 135, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_9894c391cd', 'id': 'chatcmpl-DQSlBKkgTcchxCQ8HhykgpnaTkxjj', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, 'type': 'ai', 'name': None, 'id': 'lc_run--019d5222-0091-7d42-aaa8-43784c3705bf-0', 'tool_calls': [{'name': 'multiply', 'args': {'a': 3, 'b': 2}, 'id': 'call_AE2tIpJbY3TMWImuqexrkbsY', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metad

In [ ]:
#https://docs.langchain.com/langsmith/deployment-quickstart#deploy-from-github-with-langgraph-cloud
#We can deploy to Cloud via LangSmith, as outlined above.

#Create a New Repository on GitHub

#Add Your GitHub Repository as a Remote
#Push the changes

#Connect LangSmith to your GitHub Repository
#Go to LangSmith
#Click on deployments tab on the left LangSmith panel
#Add + New Deployment
#Then, select the Github repository (e.g., langchain-academy) that you just created for the course
#Point the LangGraph API config file at one of the studio directories
#For example, for module-1 use: module-1/studio/langgraph.json
#Set your API keys (e.g., you can just copy from your module-1/studio/.env file)
#Refer https://github.com/langchain-ai/langchain-academy/blob/main/module-1/deployment.ipynb for further. Not doing as it is costly.